# Experiment 4.0.1 — Binary vs multi-spike macro-LIF

Analysis-only notebook for the 2×2 hidden/output event-cap factorial. Training is performed by the Slurm array runner.

Primary question: does the one-event-per-250-ms communication cap explain a substantial part of the current SNN decoder gap?

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

def find_repo_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'snn').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate writingRing repository root')

repo = find_repo_root()
root = repo / 'notebooks' / 'artifacts' / 'experiment_4_0_1_multispike_macro_lif' / 'macro_lif_event_cap_factorial_v1'
eval_dir = root / 'evaluations'
n_eval = len(list(eval_dir.glob('*.json'))) if eval_dir.exists() else 0
required = [root / 'runs.csv', root / 'summary.csv', root / 'factorial_effects.csv', root / 'provenance.json']
ready = n_eval == 40 and all(path.exists() for path in required)
print(f'Artifact root: {root}')
print(f'Completed evaluations: {n_eval}/40')
print(f'Finalized outputs ready: {ready}')
if n_eval == 40 and not ready:
    print('Run: sbatch scripts/bash_script/SNN_Bash/finalize_exp_4_0_1_cpu.bash')


In [ ]:
if ready:
    summary = pd.read_csv(root / 'summary.csv')
    effects = pd.read_csv(root / 'factorial_effects.csv')
    runs = pd.read_csv(root / 'runs.csv')
    display(summary)
    display(effects)
    sanity_path = root / 'binary_sanity.csv'
    if sanity_path.exists():
        print('Custom cap=1 vs original Exp4.0 binary sanity check')
        display(pd.read_csv(sanity_path))
else:
    print('Skipping finalized tables until all 40 runs and finalizer outputs are present.')


In [ ]:
if ready:
    test = runs[runs['split'] == 'test'].copy()
    order = ['binary', 'multi_h', 'multi_o', 'multi_ho']
    plot_df = (test.groupby(['architecture', 'variant'])['valid_count_balanced_accuracy']
               .agg(['mean', 'std']).reset_index())
    for architecture in ['ff', 'rsnn']:
        sub = plot_df[plot_df['architecture'] == architecture].set_index('variant').reindex(order)
        fig, ax = plt.subplots(figsize=(8, 4.5))
        ax.bar(sub.index, sub['mean'], yerr=sub['std'], capsize=4)
        ax.set_ylabel('Test balanced accuracy')
        ax.set_title(f'Exp4.0.1 {architecture.upper()}: event-cap factorial')
        ax.set_ylim(0, max(0.65, float((sub['mean'] + sub['std'].fillna(0)).max()) + 0.05))
        plt.show()


In [ ]:
if ready:
    test = runs[runs['split'] == 'test'].copy()
    diag = (test.groupby(['architecture', 'variant'])
            [['hidden_fraction_gt1', 'hidden_fraction_at_cap', 'output_fraction_gt1', 'output_fraction_at_cap']]
            .mean().reset_index())
    display(diag)
    for architecture in ['ff', 'rsnn']:
        sub = diag[diag['architecture'] == architecture].set_index('variant').reindex(['binary', 'multi_h', 'multi_o', 'multi_ho'])
        fig, ax = plt.subplots(figsize=(8, 4.5))
        ax.bar(sub.index, sub['hidden_fraction_gt1'])
        ax.set_ylabel('Fraction hidden neuron-steps with S > 1')
        ax.set_title(f'{architecture.upper()}: actual use of multi-event hidden capacity')
        plt.show()
